# CODA reproduction: 3D histology and breast IHC histomorphometry

Runs the pipeline in Colab. Two arms, and they differ in one important way.

| Arm | Data | Can it run here? |
|---|---|---|
| A | Kartasalo mouse liver, 47 serial sections | **Yes.** CC BY 4.0, downloaded by the notebook |
| C | USM breast immunohistochemistry, 234 fields | **Only if you upload it.** See below |

## Arm C is patient material, and that is a decision, not a step

The 234 immunohistochemistry fields are institutional patient images. They are
deliberately absent from the repository and have never been committed. Running
Arm C here means uploading them to Google's servers, where they are processed
and cached outside your institution.

Whether that is permitted is a question about your ethics approval and your
institution's data governance, not a technical one, and this notebook cannot
answer it for you. Arm C is therefore off by default and requires you to set a
flag confirming you have checked.

**Arm C also runs perfectly well on your own machine**, needs no GPU and takes
minutes, so uploading is rarely necessary. `python RUN_EVERYTHING.py` locally
does all of it.

Arm A uses openly licensed mouse tissue and raises none of this.

## Before you start

**Runtime, Change runtime type, T4 GPU, Save.** No stage here strictly needs a
GPU, but it makes the fetch and decode steps faster.

In [ ]:
#@title 1. Environment and code { display-mode: "form" }
import subprocess, sys, os, shutil
from pathlib import Path

REPO = 'https://github.com/swatian1989/coda-3d-histology.git'
WORK = Path('/content/coda')
if not WORK.exists():
    r = subprocess.run(['git', 'clone', '-q', REPO, str(WORK)])
    if r.returncode:
        raise SystemExit(
            'Clone failed. If the repository is private, either make it public in\n'
            'Settings, General, Change visibility, or upload the project folder to\n'
            'Colab instead.')
os.chdir(WORK)
sys.path.insert(0, str(WORK / 'src'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'tifffile', 'zarr', 'scikit-image', 'python-docx', 'tabulate'],
               check=False)
print('working dir:', os.getcwd())
print('free disk  : %.0f GB' % (shutil.disk_usage('/content').free / 1e9))
subprocess.run([sys.executable, 'RUN_EVERYTHING.py', '--list'])

In [ ]:
#@title 2. Settings and Google Drive { display-mode: "form" }
USE_DRIVE = True  #@param {type:"boolean"}
#@markdown Tick ONLY if you have confirmed that uploading institutional patient
#@markdown images to Google is permitted by your ethics approval and data
#@markdown governance. Arm C runs locally without any of this.
I_MAY_UPLOAD_PATIENT_IMAGES = False  #@param {type:"boolean"}

from pathlib import Path
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE = Path('/content/drive/MyDrive/coda_histology_results')
else:
    SAVE = Path('/content/coda_results')
SAVE.mkdir(parents=True, exist_ok=True)

for d in ('results', 'figures', 'reports', 'manuscript'):
    src, dst = Path('/content/coda') / d, SAVE / d
    dst.mkdir(parents=True, exist_ok=True)
    if src.is_symlink():
        src.unlink()
    elif src.exists():
        for f in src.rglob('*'):
            if f.is_file():
                (dst / f.relative_to(src)).parent.mkdir(parents=True, exist_ok=True)
                f.replace(dst / f.relative_to(src))
        import shutil as _s; _s.rmtree(src, ignore_errors=True)
    src.symlink_to(dst)

print('results saved to', SAVE)
print('Arm C enabled:', I_MAY_UPLOAD_PATIENT_IMAGES)

---
# Arm A: the serial liver stack

Openly licensed mouse tissue, CC BY 4.0. The archive is a single 63.79 GB zip
whose download service ignores range requests, so it cannot resume. The liver
series sits near the front, so the useful part arrives in roughly the first
15 GB and the cell below stops once all 47 sections have landed.

In [ ]:
#@title 3. Stream the liver series (stops itself at 47 sections)
import subprocess, sys, time
from pathlib import Path

SEC = Path('/content/coda/data/raw/kartasalo/extracted/Data_to_IDA/liver')
proc = subprocess.Popen([sys.executable, '-u', 'scripts/fetch_kartasalo_liver.py'],
                        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                        cwd='/content/coda')
try:
    while proc.poll() is None:
        n = len(list(SEC.glob('*.tif'))) if SEC.exists() else 0
        print(f'\r  sections: {n}/47', end='')
        if n >= 47:
            proc.terminate(); print('\n  all 47 sections present, stopping'); break
        time.sleep(20)
except KeyboardInterrupt:
    proc.terminate(); print('\n  stopped by user')
n = len(list(SEC.glob('*.tif'))) if SEC.exists() else 0
print('sections on disk:', n)
assert n >= 10, 'Too few sections. The stream did not reach the liver portion.'

In [ ]:
#@title 4. Arm A: verify, register, reconstruct, count, fibres
# Steps 2 to 7: decode check, registration with the corrected rotation
# estimator, its validation against the fiducials, the volume and the
# 2D versus 3D count, detector agreement, and fibre anisotropy.
!python RUN_EVERYTHING.py --steps 2,3,4,5,6,7 --skip-heavy

---
# Arm C: breast immunohistochemistry

**Runs only if you set the flag in cell 2.** These are patient images and this
is a governance decision, not a technical step. Arm C runs locally in minutes
with `python RUN_EVERYTHING.py --steps 8,9,10,11`, which avoids the question
entirely.

In [ ]:
#@title 5. Upload the IHC images (only if permitted)
from pathlib import Path
import zipfile, io

if not I_MAY_UPLOAD_PATIENT_IMAGES:
    print('Arm C is off. Nothing was uploaded.')
    print('Set I_MAY_UPLOAD_PATIENT_IMAGES in cell 2 only after confirming that')
    print('your ethics approval covers processing these images on Google servers.')
    print('Otherwise run Arm C locally: python RUN_EVERYTHING.py --steps 8,9,10,11')
else:
    print('Upload a ZIP of the usm folder, containing ER, PR, HER2 and KI67 subfolders.')
    from google.colab import files
    up = files.upload()
    dest = Path('/content/coda/data/raw/usm'); dest.mkdir(parents=True, exist_ok=True)
    for name, blob in up.items():
        with zipfile.ZipFile(io.BytesIO(blob)) as z:
            z.extractall(dest)
    n = len(list(dest.rglob('*.png'))) + len(list(dest.rglob('*.jpg')))
    print(f'{n} images extracted to {dest}')
    assert n > 0, 'No images found in the archive; check the folder structure.'

In [ ]:
#@title 6. Arm C: QC, markers, spatial statistics, stereology
import subprocess, sys
from pathlib import Path

if I_MAY_UPLOAD_PATIENT_IMAGES and any(Path('/content/coda/data/raw/usm').rglob('*.png')):
    subprocess.run([sys.executable, 'RUN_EVERYTHING.py',
                    '--steps', '8,9,10,11'], cwd='/content/coda')
else:
    print('Arm C skipped. Arm A is unaffected: nothing in stages 1 to 7 depends')
    print('on the immunohistochemistry data.')

In [ ]:
#@title 7. Extended Data figures, report and manuscript
!python scripts/make_extended_data_figures.py
!python RUN_EVERYTHING.py --steps 12,13,14

In [ ]:
#@title 8. Show and download the results
import json
from pathlib import Path
from IPython.display import display, Image

for p in sorted(Path('/content/coda/results/kartasalo').glob('summary*.json')):
    print(f'--- {p.name} ---')
    print(json.dumps(json.loads(p.read_text()), indent=2)[:1000], '\n')

for f in sorted(Path('/content/coda/figures').glob('F*.png'))[:8]:
    display(Image(filename=str(f)))

from google.colab import files
for f in ('reports/analysis_report.html', 'manuscript/manuscript.docx'):
    p = Path('/content/coda') / f
    if p.exists():
        print('downloading', f, f'{p.stat().st_size/1e6:.1f} MB')
        files.download(str(p))